In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import numpy as np
from PIL import Image
import torch.optim as optim
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as T
import torch.nn as nn
import segmentation_models_pytorch as smp
import random


In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO

class UnderwaterSegmentationDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None, target_transform=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.transform = transform
        self.target_transform = target_transform

        self.image_files = sorted(os.listdir(images_dir))
        self.mask_files = sorted(os.listdir(masks_dir))

        assert len(self.image_files) == len(self.mask_files), "Images and masks count mismatch"

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):

        img_path = os.path.join(self.images_dir, self.image_files[idx])
        image = Image.open(img_path).convert("RGB")

        mask_path = os.path.join(self.masks_dir, self.mask_files[idx])
        mask = Image.open(mask_path)

        if self.transform:
            image = self.transform(image)
        else:
            image = np.array(image, dtype=np.float32) / 255.0
            image = torch.from_numpy(image).permute(2, 0, 1)

        if self.target_transform:
            mask = self.target_transform(mask)
        else:
            mask = torch.from_numpy(np.array(mask)).long()

        mask = remap_mask(mask)

        return image, mask

# Define transformations
image_transform = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

mask_transform = T.Compose([
    T.Resize((256, 256), interpolation=T.InterpolationMode.NEAREST),
    T.PILToTensor()
])

images_dir = os.path.join(path, "/kaggle/input/q3-stage3-2026/dataset/images")
masks_dir  = os.path.join(path, "/kaggle/input/q3-stage3-2026/dataset/masks")

dataset = UnderwaterSegmentationDataset(images_dir, masks_dir, transform=image_transform, target_transform=mask_transform)

# Train and validation
val_ratio = 0.2
val_size = int(len(dataset) * val_ratio)
train_size = len(dataset) - val_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

#dataloader
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=8, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

In [ ]:
# Display image
def visualize_samples(dataloader, num_samples=4):
    images, masks = next(iter(dataloader))

    plt.figure(figsize=(10, 2 * num_samples))
    for i in range(num_samples):

        plt.subplot(num_samples, 2, 2*i + 1)

        img = images[i].clone()
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        img = img * std + mean
        img = torch.clip(img, 0, 1)

        plt.imshow(img.permute(1, 2, 0))
        plt.title("Image")
        plt.axis("off")

        plt.subplot(num_samples, 2, 2*i + 2)
        plt.imshow(masks[i].squeeze(), interpolation="nearest")
        plt.title("Mask")
        plt.axis("off")

    plt.tight_layout()
    plt.show()


visualize_samples(train_loader)

In [ ]:
# TO DO
!pip install segmentation-models-pytorch

NUM_CLASSES = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

#use efficientnet-b1
model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES,
    activation=None,
).to(device)

print(model.__class__.__name__)


In [ ]:
# TO DO

# Training and valid
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0

    for imgs, masks in loader:
        imgs  = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True).squeeze(1) # Remove the channel dimension

        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)

    return running_loss / len(loader.dataset)

@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0

    for imgs, masks in loader:
        imgs  = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True).squeeze(1) # Remove the channel dimension

        logits = model(imgs)
        loss = criterion(logits, masks)

        running_loss += loss.item() * imgs.size(0)

    return running_loss / len(loader.dataset)

In [ ]:
# TO DO

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

epoch = 5

train_losses, val_losses = [], []

for epoch in range(1, epoch + 1):
    tr_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    va_loss = validate_one_epoch(model, val_loader, criterion, device)

    train_losses.append(tr_loss)
    val_losses.append(va_loss)

    print(f"Epoch [{epoch}/{epoch}] | Train Loss: {tr_loss:.4f} | Val Loss: {va_loss:.4f}")

In [ ]:
# Plot loss curves
plt.figure(figsize=(7,5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.title("Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# TO DO

#Visualize
@torch.no_grad()
def visualize_predictions(model, dataset, device, n=4):
    model.eval()
    idxs = random.sample(range(len(dataset)), k=min(n, len(dataset)))

    plt.figure(figsize=(12, 4*n))
    for i, idx in enumerate(idxs):
        img, gt_mask = dataset[idx]
        x = img.unsqueeze(0).to(device)

        logits = model(x)
        pred = torch.argmax(logits, dim=1)[0]

        img_np = img.permute(1,2,0).cpu().numpy()

        plt.subplot(n, 3, 3*i + 1)
        plt.imshow(img_np)
        plt.title("Image")
        plt.axis("off")

        plt.subplot(n, 3, 3*i + 2)
        plt.imshow(gt_mask.cpu().squeeze().numpy(), interpolation="nearest")
        plt.title("Ground Truth")
        plt.axis("off")


    plt.tight_layout()
    plt.show()

visualize_predictions(model, val_dataset, device, n=4)